In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum, avg, count, broadcast
)

spark = (
    SparkSession.builder
    .appName("SparkPerformanceHandsOn")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Spark Version:", spark.version)
print("Spark session created successfully.")
print("Spark UI available at:", spark.sparkContext.uiWebUrl)

In [ ]:
print("Default parallelism:",
      spark.sparkContext.defaultParallelism)

print("Shuffle partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
from pyspark.sql.functions import rand, floor

sales_df = (
    spark.range(1, 1_000_001)
    .withColumn(
        "customer_id",
        (floor(rand() * 10000) + 1).cast("int")
    )
    .withColumn(
        "product_id",
        (floor(rand() * 100) + 1).cast("int")
    )
    .withColumn(
        "quantity",
        (floor(rand() * 10) + 1).cast("int")
    )
    .withColumn(
        "price",
        (rand() * 1000).cast("double")
    )
)

sales_df.show()
print(sales_df.count(), " records generated in sales_df.")

In [ ]:
sales_df = sales_df.withColumn(
    "sales_amount",
    col("quantity") * col("price")
)

sales_df.show(5)

In [ ]:
high_value_sales = sales_df.filter(
    col("sales_amount") > 2000
)

selected_sales = high_value_sales.select(
    "customer_id",
    "product_id",
    "quantity",
    "sales_amount"
)

In [ ]:
# selected_sales.count()
# selected_sales.show(10)
# selected_sales.explain()
selected_sales.explain("formatted")

In [ ]:
result_df = (
    sales_df
    .filter(col("sales_amount") > 1000)
    .select(
        "customer_id",
        "product_id",
        "sales_amount"
    )
    .groupBy("product_id")
    .agg(
        sum("sales_amount").alias("total_sales"),
        avg("sales_amount").alias("avg_sales"),
        count("*").alias("transactions")
    )
    .orderBy(col("total_sales").desc())
)

In [ ]:
result_df.explain("formatted")

In [ ]:
result_df.show(20)

In [ ]:
department_like_df = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("sales_amount").alias("total_sales")
    )
)

department_like_df.show()

In [ ]:
sales_df.rdd.getNumPartitions()

In [ ]:
df8 = sales_df.repartition(8)

df8.rdd.getNumPartitions()

In [ ]:
df8.groupBy("product_id").count().show()

In [ ]:
high_sales = sales_df.filter(
    col("sales_amount") > 2000
)

In [ ]:
high_sales.cache()

In [ ]:
high_sales.count()

In [ ]:
high_sales.groupBy("product_id").count().show()

In [ ]:
high_sales.is_cached

In [ ]:
high_sales.unpersist()

In [ ]:
from pyspark import StorageLevel

persisted_df = sales_df.filter(
    col("sales_amount") > 1500
)

persisted_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

In [ ]:
persisted_df.count()

In [ ]:
persisted_df.groupBy("product_id").count().show()

In [ ]:
persisted_df.unpersist()

In [ ]:
sales_by_product = sales_df.repartition(
    8,
    "product_id"
)

In [ ]:
sales_8 = sales_df.repartition(8)

print(
    sales_8.rdd.getNumPartitions()
)

In [ ]:

sales_small = sales_8.coalesce(2)

sales_small.rdd.getNumPartitions()

In [ ]:
sales_df \
    .repartition(20) \
    .coalesce(4) \
    .rdd.getNumPartitions()

In [ ]:
from pyspark.sql.functions import spark_partition_id

partition_df = sales_df.withColumn(
    "partition_id",
    spark_partition_id()
)

In [ ]:
partition_df \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

In [ ]:
products = [
    (1, "Laptop", "Electronics"),
    (2, "Mobile", "Electronics"),
    (3, "Shoes", "Fashion"),
    (4, "Watch", "Fashion"),
    (5, "Book", "Education")
]

product_df = spark.createDataFrame(
    products,
    ["product_id", "product_name", "category"]
)

product_df.show()

In [ ]:
demo_sales = sales_df.filter(
    col("product_id") <= 5
)
demo_sales.count()

In [ ]:
normal_join = demo_sales.join(
    product_df,
    "product_id"
)

In [ ]:
normal_join.explain()

In [ ]:
broadcast_join = demo_sales.join(
    broadcast(product_df),
    "product_id"
)
broadcast_join.explain()

In [ ]:
broadcast_join.show()

In [ ]:
join_without_broadcast = demo_sales.join(
    product_df,
    "product_id"
)

join_without_broadcast.explain()

In [ ]:
join_with_broadcast = demo_sales.join(
    broadcast(product_df),
    "product_id"
)

join_with_broadcast.explain()

In [ ]:
import time

start = time.perf_counter()

sales_df.groupBy("product_id").agg(
    sum("sales_amount")
).count()

end = time.perf_counter()

print(
    "Execution time:",
    round(end - start, 3),
    "seconds"
)

In [ ]:
cached_sales = sales_df.cache()

cached_sales.count()

In [70]:
start = time.perf_counter()

cached_sales.groupBy("product_id").agg(
    sum("sales_amount")
).count()

end = time.perf_counter()

print(
    "Execution time with cache:",
    round(end - start, 3),
    "seconds"
)

Execution time with cache: 0.144 seconds
